In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install networkx

In [ ]:
import subprocess
import sys
import os

def clean_install():
    """Complete clean installation with correct versions"""
    
    print("🧹 Step 1: Cleaning old packages...")
    subprocess.run([
        sys.executable, "-m", "pip", "uninstall", "-y", 
        "torch", "torchvision", "torchaudio", "transformers",
        "numpy", "scikit-learn", "tensorflow", "keras"
    ], capture_output=True)
    
    print("📦 Step 2: Installing core packages...")
    # Install numpy first with specific version
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy==1.24.3"
    ], check=True)
    
    print("🔥 Step 3: Installing PyTorch...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "torch==2.3.0", "torchvision==0.18.0",
        "--index-url", "https://download.pytorch.org/whl/cu118"
    ], check=True)
    
    print("🤖 Step 4: Installing Transformers...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers==4.44.2"
    ], check=True)
    
    print("📚 Step 5: Installing other dependencies...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "sentence-transformers", "faiss-cpu", 
        "spacy", "networkx", "datasets", "rank_bm25"
    ], check=True)
    
    print("🌐 Step 6: Downloading spacy model...")
    subprocess.run([
        sys.executable, "-m", "spacy", "download", "en_core_web_sm"
    ], capture_output=True)
    
    print("\n" + "="*70)
    print("✅ INSTALLATION COMPLETE!")
    print("="*70)
    print("🔴 IMPORTANT: RESTART RUNTIME NOW!")
    print("="*70)
    print("\nGoogle Colab: Runtime → Restart Runtime")
    print("Jupyter: Kernel → Restart Kernel")
    print("\nAfter restart, run the main code below")
    print("="*70)

# Run installation
clean_install()

In [9]:
import torch
import faiss
import numpy as np
import spacy
import networkx as nx
from typing import List, Dict, Any, Set, Tuple
import time
import pickle
import json
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from collections import defaultdict

print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")


class LightweightKG:
    """Lightweight Knowledge Graph - Optimized for Kaggle"""
    
    def __init__(self):
        self.graph = nx.DiGraph()
        self.entities = {}
        self.communities = {}
        self.entity_to_community = {}
    
    def add_entity(self, name: str, entity_type: str, source_doc: str):
        if name not in self.graph:
            self.graph.add_node(name, type=entity_type, source=source_doc)
            self.entities[name] = {'type': entity_type, 'source': source_doc}
    
    def add_relation(self, source: str, target: str):
        if source in self.graph and target in self.graph:
            self.graph.add_edge(source, target)
    
    def detect_communities_fast(self):
        """Fast community detection - won't hang!"""
        if self.graph.number_of_nodes() == 0:
            print("⚠️  Empty graph, skipping communities")
            return
        
        num_nodes = self.graph.number_of_nodes()
        print(f"🔍 Detecting communities ({num_nodes} nodes)...")
        
        # Use fast algorithm based on size
        if num_nodes > 3000:
            print("   Graph too large, using simple type-based clustering")
            self._simple_clustering()
        elif num_nodes > 800:
            print("   Using fast label propagation...")
            undirected = self.graph.to_undirected()
            from networkx.algorithms import community
            communities_gen = community.label_propagation_communities(undirected)
            self._store_communities(list(communities_gen))
        else:
            print("   Using greedy modularity...")
            undirected = self.graph.to_undirected()
            from networkx.algorithms import community
            communities_gen = community.greedy_modularity_communities(undirected)
            self._store_communities(list(communities_gen))
        
        print(f"✅ Detected {len(self.communities)} communities")
    
    def _store_communities(self, communities_list):
        self.communities = {}
        self.entity_to_community = {}
        for comm_id, comm in enumerate(communities_list):
            members = list(comm)
            self.communities[comm_id] = members
            for entity in members:
                self.entity_to_community[entity] = comm_id
    
    def _simple_clustering(self):
        """Fast type-based clustering"""
        type_clusters = defaultdict(list)
        for entity, metadata in self.entities.items():
            entity_type = metadata.get('type', 'UNKNOWN')
            type_clusters[entity_type].append(entity)
        
        self.communities = {}
        self.entity_to_community = {}
        for comm_id, (entity_type, members) in enumerate(type_clusters.items()):
            self.communities[comm_id] = members
            for entity in members:
                self.entity_to_community[entity] = comm_id
    
    def local_retrieval(self, query_entities: List[str]) -> List[str]:
        """Get entity + neighbors"""
        expanded = set(query_entities)
        for entity in query_entities:
            if entity in self.graph:
                neighbors = set(self.graph.successors(entity)) | set(self.graph.predecessors(entity))
                expanded.update(neighbors)
        return list(expanded)
    
    def global_retrieval(self, query_entities: List[str]) -> List[str]:
        """Get community members"""
        community_entities = set()
        for entity in query_entities:
            if entity in self.entity_to_community:
                comm_id = self.entity_to_community[entity]
                community_entities.update(self.communities[comm_id])
        return list(community_entities)
    
    def hybrid_retrieval(self, query_entities: List[str]) -> List[str]:
        """Combined local + global"""
        local = set(self.local_retrieval(query_entities))
        global_ents = set(self.global_retrieval(query_entities))
        return list(local | global_ents)
    
    def save(self, filepath: str):
        data = {
            'graph': nx.node_link_data(self.graph),
            'entities': self.entities,
            'communities': self.communities,
            'entity_to_community': self.entity_to_community
        }
        with open(filepath, 'wb') as f:
            pickle.dump(data, f)
        print(f"💾 KG saved ({os.path.getsize(filepath) / 1024 / 1024:.1f} MB)")
    
    def load(self, filepath: str) -> bool:
        if not os.path.exists(filepath):
            return False
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        self.graph = nx.node_link_graph(data['graph'])
        self.entities = data['entities']
        self.communities = data['communities']
        self.entity_to_community = data['entity_to_community']
        print(f"✅ KG loaded: {self.graph.number_of_nodes()} nodes, "
              f"{len(self.communities)} communities")
        return True

import sqlite3

import sqlite3
import os
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
import networkx as nx

class PersistentKG:
    """Persistent Knowledge Graph with vector embeddings for semantic reasoning."""

    def __init__(self, db_path="local_kg.db", embedder_model="all-mpnet-base-v2"):
        self.db_path = db_path
        self.graph = nx.DiGraph()
        self.entities = {}
        self.communities = {}
        self.entity_to_community = {}
        self.embedder = SentenceTransformer(embedder_model)
        self._init_db()

    def _init_db(self):
        conn = sqlite3.connect(self.db_path)
        cur = conn.cursor()
        cur.execute("""
        CREATE TABLE IF NOT EXISTS entities (
            name TEXT PRIMARY KEY,
            type TEXT,
            source_doc TEXT
        );
        """)
        cur.execute("""
        CREATE TABLE IF NOT EXISTS relations (
            source TEXT,
            target TEXT
        );
        """)
        cur.execute("""
        CREATE TABLE IF NOT EXISTS entity_embeddings (
            name TEXT PRIMARY KEY,
            embedding BLOB
        );
        """)
        conn.commit()
        conn.close()

    def add_entity(self, name: str, entity_type: str, source_doc: str):
        """Add entity and compute embedding if not present."""
        if not name or name in self.entities:
            return
        self.entities[name] = {'type': entity_type, 'source': source_doc}
        self.graph.add_node(name, type=entity_type, source=source_doc)

        # Store in DB
        conn = sqlite3.connect(self.db_path)
        cur = conn.cursor()
        cur.execute("""
            INSERT OR IGNORE INTO entities (name, type, source_doc)
            VALUES (?, ?, ?)
        """, (name, entity_type, source_doc))
        conn.commit()
        conn.close()

        # Create & store embedding
        self.add_embedding(name)

    def add_relation(self, source: str, target: str):
        """Add directed relation."""
        if not source or not target:
            return
        self.graph.add_edge(source, target)
        conn = sqlite3.connect(self.db_path)
        cur = conn.cursor()
        cur.execute("""
            INSERT OR IGNORE INTO relations (source, target)
            VALUES (?, ?)
        """, (source, target))
        conn.commit()
        conn.close()

    def add_embedding(self, name: str):
        """Compute & store embedding for an entity."""
        emb = self.embedder.encode(name).astype(np.float32)
        emb_bytes = pickle.dumps(emb)
        conn = sqlite3.connect(self.db_path)
        cur = conn.cursor()
        cur.execute("""
            INSERT OR REPLACE INTO entity_embeddings (name, embedding)
            VALUES (?, ?)
        """, (name, emb_bytes))
        conn.commit()
        conn.close()

    def get_embedding(self, name: str):
        """Retrieve stored embedding from DB."""
        conn = sqlite3.connect(self.db_path)
        cur = conn.cursor()
        cur.execute("SELECT embedding FROM entity_embeddings WHERE name=?", (name,))
        row = cur.fetchone()
        conn.close()
        if not row:
            return None
        return pickle.loads(row[0])

    def semantic_search(self, query: str, top_k=5):
        """Return top_k semantically similar entities."""
        q_emb = self.embedder.encode(query).astype(np.float32)
        results = []
        conn = sqlite3.connect(self.db_path)
        cur = conn.cursor()
        cur.execute("SELECT name, embedding FROM entity_embeddings")
        for name, blob in cur.fetchall():
            emb = pickle.loads(blob)
            score = np.dot(q_emb, emb) / (np.linalg.norm(q_emb) * np.linalg.norm(emb))
            results.append((name, float(score)))
        conn.close()
        results.sort(key=lambda x: x[1], reverse=True)
        return results[:top_k]

    def load_from_db(self):
        """Rebuild graph and entities from SQLite."""
        if not os.path.exists(self.db_path):
            print("⚠️ No persistent KG found.")
            return False
        conn = sqlite3.connect(self.db_path)
        cur = conn.cursor()
        cur.execute("SELECT name, type, source_doc FROM entities")
        for name, etype, src in cur.fetchall():
            self.graph.add_node(name, type=etype, source=src)
            self.entities[name] = {'type': etype, 'source': src}
        cur.execute("SELECT source, target FROM relations")
        for s, t in cur.fetchall():
            self.graph.add_edge(s, t)
        conn.close()
        print(f"✅ Loaded KG with {len(self.entities)} entities and {len(self.graph.edges())} relations")
        return True

    def local_retrieval(self, query_entities, max_neighbors=3):

        """

        Local graph expansion: get immediate neighbors for given entities.

        """

        expanded = set(query_entities)

        for ent in query_entities:

            if ent in self.graph:

                neighbors = list(self.graph.neighbors(ent))[:max_neighbors]

                expanded.update(neighbors)

        return list(expanded)



    def global_retrieval(self, query_entities, top_k=5):

        """

        Global reasoning: if communities or embeddings exist, return similar nodes.

        """

        results = []

        if hasattr(self, "semantic_search"):

            for ent in query_entities:

                similar = self.semantic_search(ent, top_k=top_k)

                results.extend([name for name, _ in similar])

        return list(set(results + query_entities))



    def hybrid_retrieval(self, query_entities, top_k=5):

        """

        Combines local neighborhood and global semantic expansion.

        """

        local = self.local_retrieval(query_entities)

        global_ents = self.global_retrieval(query_entities, top_k=top_k)

        combined = list(set(local + global_ents))

        return combined

    def load(self, filepath: str = None):

        """

        Compatibility wrapper for PersistentKG to behave like LightweightKG.load().

        Ignores 'filepath' since PersistentKG data lives in SQLite.

        """

        print("ℹ️ Using persistent KG (SQLite). Loading from DB...")

        return self.load_from_db()


class KaggleLightRAG:
    """Kaggle-Optimized LightRAG - Enhanced with Quality Controls"""
    
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"🚀 Device: {self.device}")
        
        # Load spaCy
        print("🔄 Loading spaCy (lightweight)...")
        try:
            self.nlp = spacy.load("en_core_web_sm", disable=['parser', 'textcat'])
            print("✅ spaCy loaded")
        except:
            print("⚠️  spaCy not available, using fallback")
            self.nlp = None
        
        # Load models
        print("🔄 Loading embedding model...")
        self.embedder = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
        
        print("🔄 Loading T5 generator...")
        self.tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.generator = AutoModelForSeq2SeqLM.from_pretrained(
            "google/flan-t5-base",
            torch_dtype=dtype,
            device_map="auto" if self.device == "cuda" else None,
            low_cpu_mem_usage=True
        )
        if self.device == "cpu":
            self.generator = self.generator.to(self.device)
        
        # Data structures
        self.documents = []
        self.titles = []
        self.index = None
        self.bm25 = None
        self.kg = PersistentKG()
        self.kg.load_from_db()  # auto reload if file exists
        print("✅ Models loaded")
    
    def extract_entities(self, text: str, limit: int = 6) -> List[Tuple[str, str]]:
        """Extract entities - conservative to keep graph small"""
        entities = []
        
        if self.nlp:
            doc = self.nlp(text[:500])  # Process only first 500 chars
            seen = set()
            
            for ent in doc.ents:
                if ent.label_ in ("PERSON", "ORG", "GPE", "PRODUCT"):
                    if ent.text not in seen and len(ent.text) > 2:
                        entities.append((ent.text, ent.label_))
                        seen.add(ent.text)
                        if len(entities) >= limit:
                            break
        
        return entities
    
    def load_documents(self, num_docs: int = 2000, priority_topics: Set[str] = None):
        """Load Wikipedia documents - Kaggle optimized"""
        print(f"\n📥 Loading {num_docs} documents...")
        
        if priority_topics is None:
            priority_topics = set()
        
        dataset = load_dataset("wikimedia/wikipedia", "20231101.en",
                             split="train", streaming=True)
        
        count = 0
        max_attempts = num_docs * 15
        
        for item in dataset:
            if len(self.documents) >= num_docs:
                break
            if count >= max_attempts:
                break
            count += 1
            
            if 'text' not in item or 'title' not in item:
                continue
            
            title, text = item['title'], item['text']
            
            if len(text) < 300 or "may refer to:" in text[:500]:
                continue
            
            # Check priority
            is_priority = any(topic.lower() in title.lower() 
                            for topic in priority_topics)
            
            if is_priority or len(self.documents) < num_docs * 0.4:
                self.documents.append(text)
                self.titles.append(title)
                
                if len(self.documents) % 500 == 0:
                    print(f"   📖 {len(self.documents)} documents loaded")
        
        print(f"✅ Loaded {len(self.documents)} documents")
        
        # Build indices
        self._build_indices()
        
        # Build KG (limited to keep fast)
        if self.nlp:
            self._build_kg_lightweight()
    
    def _build_indices(self):
        """Build FAISS and BM25"""
        print("🔍 Building indices...")
        
        # FAISS
        texts = [f"{t}: {d[:500]}" for t, d in zip(self.titles, self.documents)]
        embeddings = self.embedder.encode(texts, show_progress_bar=True, batch_size=32)
        
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings.astype(np.float32))
        
        # BM25
        tokenized_docs = [doc.lower().split() for doc in self.documents]
        self.bm25 = BM25Okapi(tokenized_docs)
        
        print(f"✅ Indices built: {self.index.ntotal} vectors")
    
    def _build_kg_lightweight(self):
        """Build KG - LIGHTWEIGHT to avoid hanging"""
        print("\n🔨 Building Knowledge Graph (lightweight mode)...")
        
        # CRITICAL: Limit to 1500 docs max to keep graph manageable
        docs_to_process = min(len(self.documents), 1500)
        
        print(f"   Processing {docs_to_process} docs (keeping graph small)")
        
        for idx in range(docs_to_process):
            if idx % 300 == 0 and idx > 0:
                print(f"   Processed {idx}/{docs_to_process}...")
            
            doc = self.documents[idx]
            title = self.titles[idx]
            
            # Extract fewer entities (6 instead of 12)
            text = f"{title}. {doc[:500]}"
            entities = self.extract_entities(text, limit=6)
            
            # Add to graph
            for ent_name, ent_type in entities:
                self.kg.add_entity(ent_name, ent_type, title)
            
            # Add fewer relationships (only 2 per entity)
            entity_names = [e[0] for e in entities]
            for i, e1 in enumerate(entity_names):
                for e2 in entity_names[i+1:i+3]:  # Only 2 connections
                    if e1 != e2:
                        self.kg.add_relation(e1, e2)
        
        print(f"✅ KG built: {self.kg.graph.number_of_nodes()} nodes, "
              f"{self.kg.graph.number_of_edges()} edges")
        
        # Fast community detection
        self.kg.detect_communities_fast()
    
    def retrieve(self, query: str, k: int = 8, mode: str = "hybrid") -> List[Dict]:
        """Retrieve with hybrid (dense + sparse + KG semantic) fusion."""
        
        # Extract entities
        query_entities = []
        if self.nlp:
            entities = self.extract_entities(query, limit=5)
            query_entities = [e[0] for e in entities]
        
        if not query_entities:
            query_entities = [w for w in query.split() if len(w) > 3][:3]
        
        # Expand via KG (if available)
        expanded_entities = query_entities.copy()
        if mode == "local" and self.kg.graph.number_of_nodes() > 0:
            expanded_entities = self.kg.local_retrieval(query_entities)
        elif mode == "global" and self.kg.communities:
            expanded_entities = self.kg.global_retrieval(query_entities)
        elif mode == "hybrid" and self.kg.graph.number_of_nodes() > 0:
            expanded_entities = self.kg.hybrid_retrieval(query_entities)
        
        # Generate KG semantic expansion (new feature!)
        semantic_entities = []
        if hasattr(self.kg, "semantic_search"):
            sem_results = self.kg.semantic_search(query, top_k=5)
            semantic_entities = [name for name, _ in sem_results]
        
        # Combine all signals
        queries = [
            query,
            ' '.join(query_entities),
            ' '.join(expanded_entities[:5]),
            ' '.join(semantic_entities[:5])
        ]
        
        combined = {}
        
        for idx, q_variant in enumerate(queries):
            weight = 1.0 if idx == 0 else 0.5
        
            # Dense (FAISS)
            q_emb = self.embedder.encode([q_variant])
            faiss.normalize_L2(q_emb)
            dense_scores, dense_indices = self.index.search(q_emb.astype(np.float32), k * 2)
        
            for doc_idx, score in zip(dense_indices[0], dense_scores[0]):
                combined[doc_idx] = combined.get(doc_idx, 0.0) + float(score) * weight * 0.6
        
            # Sparse (BM25)
            sparse_scores = self.bm25.get_scores(q_variant.lower().split())
            sparse_indices = np.argsort(sparse_scores)[-k * 2:][::-1]
            max_sparse = max(sparse_scores) if max(sparse_scores) > 0 else 1.0
        
            for doc_idx in sparse_indices:
                norm_score = sparse_scores[doc_idx] / max_sparse
                combined[doc_idx] = combined.get(doc_idx, 0.0) + norm_score * weight * 0.4
        
        # Sort & return
        ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:k]
        
        return [{
            'content': self.documents[idx],
            'title': self.titles[idx],
            'score': float(score)
        } for idx, score in ranked]
        
    def generate(self, query: str, docs: List[Dict]) -> str:
        """
        Generate detailed answer from context with quality controls
        NEW: Enhanced from EnhancedRAG with longer answers and post-processing
        """
        if not docs:
            return "I couldn't find relevant information to answer this question."
        
        # Use top 5 docs with more content (1000 chars instead of 800)
        context_parts = []
        for i, doc in enumerate(docs[:5], 1):
            content = doc['content'][:1000].replace('\n', ' ').strip()
            context_parts.append(f"Source {i} ({doc['title']}):\n{content}")
        
        context = "\n\n".join(context_parts)
        
        # Enhanced prompt for more comprehensive answers
        prompt = f"""You are a knowledgeable assistant. Based on the provided sources, give a comprehensive answer to the question. Include relevant details and explain clearly. Aim for 3-4 complete sentences, ensuring the response ends with a full stop.

Sources:
{context}

Question: {query}

Provide a detailed answer (3-4 complete sentences):"""
        
        inputs = self.tokenizer(prompt, return_tensors="pt",
                               max_length=2048,  # Increased from 1536
                               truncation=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.generator.generate(
                **inputs,
                max_length=300,      # Increased from 250
                min_length=60,       # Increased from 50
                num_beams=6,         # Increased from 4 for better quality
                length_penalty=1.5,  # Increased from 1.2
                early_stopping=True,
                no_repeat_ngram_size=3,
                temperature=0.8,     # Added for diversity
                do_sample=False
            )
        
        answer = self.tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        
        # NEW: Post-processing to ensure quality
        if len(answer) < 50:
            # Fallback: extract sentences from first document
            print("   ⚠️  Generated answer too short, using document extraction")
            first_doc = docs[0]['content']
            sentences = [s.strip() for s in first_doc.split('.') if len(s.strip()) > 30]
            if sentences:
                answer = '. '.join(sentences[:3]) + '.'
        else:
            # Ensure answer ends with complete sentence
            sentences = answer.split('.')
            sentences = [s.strip() for s in sentences if s.strip()]
            if len(sentences) > 3:
                answer = '. '.join(sentences[:max(3, len(sentences))]) + '.'
            else:
                answer = '. '.join(sentences) + '.'
        
        return answer
    
    def query(self, question: str, k: int = 8, mode: str = "hybrid", 
              score_threshold: float = 0.2) -> Dict:
        """
        Complete query pipeline with relevance filtering
        NEW: Added relevance threshold check from EnhancedRAG
        """
        start = time.time()
        
        print(f"\n❓ {question}")
        print(f"🎯 Mode: {mode.upper()}")
        
        docs = self.retrieve(question, k=k, mode=mode)
        
        print(f"   📚 Retrieved {len(docs)} documents")
        for i, doc in enumerate(docs[:4], 1):
            title_preview = doc['title'][:50] + "..." if len(doc['title']) > 50 else doc['title']
            print(f"      {i}. {title_preview} (score: {doc['score']:.3f})")
        
        # NEW: Relevance filtering
        top_doc_score = docs[0]['score'] if docs else 0.0
        
        if top_doc_score >= score_threshold:
            relevance_score = top_doc_score
            print(f"   ✅ Relevance check: Top doc score ({top_doc_score:.3f}) meets threshold ({score_threshold})")
            answer = self.generate(question, docs)
        else:
            # Check average score as fallback
            avg_score = sum(doc['score'] for doc in docs) / len(docs) if docs else 0.0
            relevance_score = avg_score
            
            if avg_score < score_threshold:
                answer = "We don't have enough context on this question."
                print(f"   ⚠️  Relevance check: Avg score ({avg_score:.3f}) below threshold ({score_threshold})")
            else:
                answer = self.generate(question, docs)
                print(f"   ✅ Relevance check: Avg score ({avg_score:.3f}) meets threshold ({score_threshold})")
        
        elapsed = time.time() - start
        
        print(f"   ✅ ANSWER: {answer}")
        print(f"   📏 Length: {len(answer)} chars, {len(answer.split())} words")
        print(f"   📊 Relevance Score: {relevance_score:.3f}")
        print(f"   ⏱️  {elapsed:.2f}s")
        print("-" * 70)
        
        return {
            'question': question,
            'answer': answer,
            'mode': mode,
            'sources': [d['title'] for d in docs[:4]],
            'time': elapsed,
            'answer_length': len(answer),
            'relevance_score': relevance_score
        }
    
    def save(self, save_dir: str = "./rag_kaggle"):
        """Save system"""
        print(f"\n💾 Saving to {save_dir}...")
        os.makedirs(save_dir, exist_ok=True)
        
        # Documents
        with open(f"{save_dir}/documents.json", 'w', encoding='utf-8') as f:
            json.dump({'documents': self.documents, 'titles': self.titles}, f)
        
        # FAISS
        if self.index:
            faiss.write_index(self.index, f"{save_dir}/faiss.index")
        
        # KG
        self.kg.save(f"{save_dir}/kg.pkl")
        
        # Metadata
        with open(f"{save_dir}/metadata.json", 'w') as f:
            json.dump({
                'docs': len(self.documents),
                'kg_nodes': self.kg.graph.number_of_nodes(),
                'saved_at': time.strftime("%Y-%m-%d %H:%M:%S")
            }, f)
        
        print("✅ Saved successfully!")
    
    def load(self, save_dir: str = "./rag_kaggle") -> bool:
        """Load system"""
        if not os.path.exists(save_dir):
            return False
        
        print(f"\n📂 Loading from {save_dir}...")
        
        # Documents
        with open(f"{save_dir}/documents.json", 'r', encoding='utf-8') as f:
            data = json.load(f)
            self.documents = data['documents']
            self.titles = data['titles']
        
        # FAISS
        self.index = faiss.read_index(f"{save_dir}/faiss.index")
        
        # BM25
        tokenized_docs = [doc.lower().split() for doc in self.documents]
        self.bm25 = BM25Okapi(tokenized_docs)
        
        # KG
        self.kg.load(f"{save_dir}/kg.pkl")
        
        print(f"✅ Loaded: {len(self.documents)} docs, {self.kg.graph.number_of_nodes()} nodes")
        return True


def main():
    """Enhanced main function with performance tracking"""
    print("\n" + "="*70)
    print("🚀 ENHANCED KAGGLE LIGHTRAG")
    print("   With Relevance Filtering & Detailed Answers")
    print("="*70 + "\n")
    
    # Test queries
    test_queries = [
        "What is artificial intelligence?",
        "Who was Albert Einstein?",
        "How does photosynthesis work?",
        "What is machine learning?",
        "What is the capital of France?",
        "Explain quantum mechanics",
        "What is DNA?",
        "What is blockchain technology?"
    ]
    
    rag = KaggleLightRAG()
    
    # Try to load, build if not found
    save_dir = "/kaggle/working/rag_model"  # Change to "./rag_model" for local
    
    if rag.load(save_dir):
        print("\n✅ Loaded existing model!")
    else:
        print("\n🔧 Building new model...")
        
        # Extract priorities
        priority_topics = {"Artificial intelligence", "Einstein", "Photosynthesis", 
                          "Machine learning", "Physics", "Biology", "Computer",
                          "France", "Paris", "Quantum", "DNA", "Blockchain"}
        
        # KAGGLE OPTIMIZED: Only 2000 docs (fast!)
        rag.load_documents(num_docs=2000, priority_topics=priority_topics)
        
        # Save for future runs
        rag.save(save_dir)
    
    # Test with mode comparison on first query
    print("\n" + "="*70)
    print("🧪 TESTING LIGHTRAG MODES (First Query)")
    print("="*70)
    
    test_q = test_queries[0]
    mode_results = []
    
    for mode in ["naive", "local", "global", "hybrid"]:
        print(f"\n{'='*70}")
        print(f"MODE: {mode.upper()}")
        print("="*70)
        result = rag.query(test_q, mode=mode, score_threshold=0.2)
        mode_results.append(result)
    
    # Run other queries with hybrid mode
    print("\n" + "="*70)
    print("🧪 OTHER QUERIES (HYBRID MODE + RELEVANCE FILTERING)")
    print("="*70)
    
    results = []
    for q in test_queries[1:]:
        result = rag.query(q, mode="hybrid", score_threshold=0.2)
        results.append(result)
    
    # Performance summary
    print("\n" + "="*70)
    print("📊 PERFORMANCE SUMMARY")
    print("="*70)
    
    all_results = mode_results + results
    successful = sum(1 for r in all_results 
                    if len(r['answer']) > 50 
                    and "We don't have enough context" not in r['answer'])
    
    avg_time = sum(r['time'] for r in all_results) / len(all_results)
    avg_length = sum(r['answer_length'] for r in all_results) / len(all_results)
    avg_relevance = sum(r['relevance_score'] for r in all_results) / len(all_results)
    
    print(f"✅ Successful queries: {successful}/{len(all_results)}")
    print(f"⏱️  Average time: {avg_time:.2f}s")
    print(f"📏 Average answer length: {avg_length:.0f} chars")
    print(f"📊 Average relevance score: {avg_relevance:.3f}")
    print(f"📚 Total documents: {len(rag.documents)}")
    print(f"🧠 KG Nodes: {rag.kg.graph.number_of_nodes()}")
    print(f"🔗 KG Edges: {rag.kg.graph.number_of_edges()}")
    print(f"👥 Communities: {len(rag.kg.communities)}")
    print(f"💾 Device: {rag.device}")
    
    # Sample answers
    print("\n" + "="*70)
    print("📝 SAMPLE DETAILED ANSWERS")
    print("="*70)
    
    for i, r in enumerate(results[:5], 1):
        print(f"\n{i}. Q: {r['question']}")
        print(f"   A: {r['answer']}")
        print(f"   📖 Sources: {', '.join(r['sources'][:3])}")
        print(f"   📏 {r['answer_length']} chars, {len(r['answer'].split())} words")
        print(f"   📊 Relevance: {r['relevance_score']:.3f}")
        print(f"   🎯 Mode: {r['mode']}")
    
    print("\n" + "="*70)
    print("✅ COMPLETE!")
    print("="*70)
    print("\n✨ System ready!")
    print("Usage: rag.query('your question', mode='hybrid', score_threshold=0.2)")
    
    return rag


if __name__ == "__main__":
    rag = main()

✅ PyTorch: 2.3.0+cu118
✅ CUDA: True

🚀 ENHANCED KAGGLE LIGHTRAG
   With Relevance Filtering & Detailed Answers

🚀 Device: cuda
🔄 Loading spaCy (lightweight)...
✅ spaCy loaded
🔄 Loading embedding model...
🔄 Loading T5 generator...


/usr/local/lib/python3.11/dist-packages/accelerate/utils/modeling.py:1614: UserWarning: The following device_map keys do not match any submodules in the model: ['decoder.embed_tokens']
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Loaded KG with 1276 entities and 1111 relations
✅ Models loaded

📂 Loading from /kaggle/working/rag_model...
ℹ️ Using persistent KG (SQLite). Loading from DB...
✅ Loaded KG with 1276 entities and 1111 relations
✅ Loaded: 912 docs, 1276 nodes

✅ Loaded existing model!

🧪 TESTING LIGHTRAG MODES (First Query)

MODE: NAIVE

❓ What is artificial intelligence?
🎯 Mode: NAIVE


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

   📚 Retrieved 8 documents
      1. Artificial intelligence (score: 0.850)
      2. Computer science (score: 0.839)
      3. Anarcho-capitalism (score: 0.600)
      4. Aesthetics (score: 0.595)
   ✅ Relevance check: Top doc score (0.850) meets threshold (0.2)
   ✅ ANSWER: The fields of cryptography and computer security involve studying the means for secure communication and for preventing security vulnerabilities. Computer graphics and computational geometry address the generation of images. Programming language theory considers different ways to describe computational processes, and database theory concerns the management of repositories of data.
   📏 Length: 383 chars, 49 words
   📊 Relevance Score: 0.850
   ⏱️  5.57s
----------------------------------------------------------------------

📊 PERFORMANCE SUMMARY
✅ Successful queries: 11/11
⏱️  Average time: 7.64s
📏 Average answer length: 419 chars
📊 Average relevance score: 1.541
📚 Total documents: 912
🧠 KG Nodes: 1276
🔗 KG Edges: 111

In [4]:
!cp local_kg.db /kaggle/working/

cp: 'local_kg.db' and '/kaggle/working/local_kg.db' are the same file


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
